# 2. Clustering 1910 CLIP Embeddings via DBSCAN

Inputs:
- 1910_embeddings.npy
- 1910_embeddings.txt (row-aligned with .npy)

Outputs:
- 1910_clusters.json (only cluster_id -> list[str] of .jpg paths)
- 1910_metadata.json
- 1910_cluster_stats.json

## Setup

In [8]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.cluster import DBSCAN

import matplotlib.pyplot as plt

## Load filenames and embeddings

In [9]:
# Paths
EMB_NPY = Path("1910_embeddings.npy")
EMB_TXT = Path("1910_embeddings.txt")
PHOTOS_DIR = Path("1910_photos")

print("Using PHOTOS_DIR:", PHOTOS_DIR)

Using PHOTOS_DIR: 1910_photos


In [10]:
filenames_1910 = [ln.strip() for ln in EMB_TXT.read_text(encoding="utf-8").splitlines() if ln.strip()]
print("filenames:", len(filenames_1910))

filenames: 72062


In [11]:
embeddings_1910 = np.load(EMB_NPY)
print("embeddings:", embeddings_1910.shape)

if embeddings_1910.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape={embeddings_1910.shape}")

if len(filenames_1910) != embeddings_1910.shape[0]:
    raise ValueError(
        "Row mismatch: filenames and embedding rows must match. "
        f"Got {len(filenames_1910)} filenames vs {embeddings_1910.shape[0]} embedding rows."
    )

print("Alignment check passed.")

embeddings: (72062, 512)
Alignment check passed.


## DBSCAN

In [14]:
eps = 2.4
min_samples = 2

start = time.time()
db = DBSCAN(eps=2.4, min_samples=2, n_jobs=-1).fit(embeddings_1910)
end = time.time()
elapsed = end - start
print(f'Time taken: {elapsed:.6f} seconds')

labels = db.labels_

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

Time taken: 30.446741 seconds
Estimated number of clusters: 2321
Estimated number of noise points: 65941


In [15]:
# determine cluster sizes and sort by size

cluster_sizes = []
for i in range(0, n_clusters_):
    cluster_sizes.append(np.sum(labels == i))
    
cluster_order = np.argsort(np.array(cluster_sizes))[::-1]

# print sizes of clusters
for i in range(0, n_clusters_):
    print("Size of cluster: " + str(cluster_order[i]) + ": " + str(np.sum(labels == cluster_order[i])))

Size of cluster: 84: 106
Size of cluster: 88: 62
Size of cluster: 4: 62
Size of cluster: 39: 57
Size of cluster: 125: 41
Size of cluster: 89: 32
Size of cluster: 90: 32
Size of cluster: 94: 32
Size of cluster: 134: 28
Size of cluster: 561: 21
Size of cluster: 193: 20
Size of cluster: 589: 19
Size of cluster: 167: 18
Size of cluster: 247: 18
Size of cluster: 213: 17
Size of cluster: 712: 17
Size of cluster: 124: 15
Size of cluster: 356: 15
Size of cluster: 774: 14
Size of cluster: 135: 14
Size of cluster: 305: 14
Size of cluster: 105: 14
Size of cluster: 162: 14
Size of cluster: 730: 13
Size of cluster: 631: 13
Size of cluster: 307: 13
Size of cluster: 106: 12
Size of cluster: 1694: 12
Size of cluster: 327: 11
Size of cluster: 541: 11
Size of cluster: 704: 11
Size of cluster: 538: 10
Size of cluster: 80: 10
Size of cluster: 473: 10
Size of cluster: 266: 10
Size of cluster: 111: 10
Size of cluster: 333: 10
Size of cluster: 706: 9
Size of cluster: 650: 9
Size of cluster: 437: 9
Size of cl

## Build 1910_cluster.json

In [16]:
# save cluster JSON 
    
# quick function for isolating cluster
def fetch_cluster(cluster_id):
    cluster = []

    cluster_mask = (labels == cluster_id)
    for i in range(0, len(cluster_mask)):
        if cluster_mask[i]:
            cluster.append(filenames_1910[i])
            
    return cluster

cluster_dictionary = {}

for i in range(0, n_clusters_):
        
    index = cluster_order[i]
    cluster = fetch_cluster(index)
    cluster_dictionary[i] = cluster
    
with open('1910_clusters.json', 'w') as fp:
    json.dump(cluster_dictionary, fp)